# AgentArk Online API Evaluation Tutorial

This Colab runs a single AgentArk task with an OpenAI-compatible online model provider. It clones AgentArk from GitHub, downloads the Linux runtime, and reads your API key from either a temporary form value or a Colab Secret/environment variable.

For public notebooks, prefer Colab Secrets and avoid saving a notebook after typing a real key into `API_KEY`.


In [ ]:
# @title Step 1: Install AgentArk and download the Linux runtime { display-mode: "form" }
# @markdown Clones the public AgentArk repository, creates a Python 3.10 virtual environment, downloads the Unity runtime, and installs AgentArk in editable mode.

AGENTARK_REPO_URL = "https://github.com/P90-RushB/AgentArk.git"  # @param {type:"string"}
AGENTARK_BRANCH = "main"  # @param {type:"string"}
ENV_ZIP_URL = "https://huggingface.co/datasets/P90-RushB/AgentArk/resolve/main/artifacts/envs/1.0.1/linux/AgentArk-env-1.0.1-linux.zip?download=true"  # @param {type:"string"}
FORCE_REINSTALL = False  # @param {type:"boolean"}

import shutil
import subprocess
import sys
from pathlib import Path

%cd /content

CONTENT_ROOT = Path("/content")
AGENTARK_ROOT = CONTENT_ROOT / "agent-ark"
VENV_DIR = CONTENT_ROOT / "agentark_env"
VENV_PY = VENV_DIR / "bin" / "python"
ENV_ZIP = CONTENT_ROOT / "AgentArk-env-1.0.1-linux.zip"
ENV_ROOT = CONTENT_ROOT / "AgentArk-env-1.0.1-linux"


def run(cmd):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)


print("Installing OS packages...")
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "git", "unzip", "wget", "xvfb", "python3.10", "python3.10-venv"])

if FORCE_REINSTALL:
    print("Removing previous /content installation...")
    shutil.rmtree(AGENTARK_ROOT, ignore_errors=True)
    shutil.rmtree(VENV_DIR, ignore_errors=True)
    shutil.rmtree(ENV_ROOT, ignore_errors=True)
    if ENV_ZIP.exists():
        ENV_ZIP.unlink()

if not VENV_DIR.exists():
    run(["python3.10", "-m", "venv", VENV_DIR])
else:
    print("Using existing virtual environment:", VENV_DIR)

if not AGENTARK_ROOT.exists():
    branch = AGENTARK_BRANCH.strip()
    clone_cmd = ["git", "clone", "--depth", "1"]
    if branch:
        clone_cmd.extend(["--branch", branch])
    clone_cmd.extend([AGENTARK_REPO_URL, AGENTARK_ROOT])
    run(clone_cmd)
else:
    print("Using existing AgentArk checkout:", AGENTARK_ROOT)

if not (AGENTARK_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(f"Expected {AGENTARK_ROOT / 'pyproject.toml'} after cloning AgentArk")

if not ENV_ROOT.exists():
    run(["wget", "-q", "-O", ENV_ZIP, ENV_ZIP_URL])
    run(["unzip", "-q", "-o", ENV_ZIP, "-d", CONTENT_ROOT])
else:
    print("Using existing Unity runtime:", ENV_ROOT)

exe = ENV_ROOT / "AgentArk.x86_64"
if not exe.exists():
    raise FileNotFoundError(f"Expected Unity executable at {exe}")
exe.chmod(0o755)

print("Installing AgentArk Python package...")
run([VENV_PY, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([VENV_PY, "-m", "pip", "install", "-q", "-e", AGENTARK_ROOT])

print("Installing notebook helper packages...")
run([sys.executable, "-m", "pip", "install", "-q", "omegaconf"])

print("AgentArk root:", AGENTARK_ROOT)
print("Unity runtime:", ENV_ROOT)
print("Python:", VENV_PY)


## Configure Runtime And Preview Tasks

Step 2 writes `/content/agent-ark/.env`, enables the Colab virtual display mode in the Unity runtime config, and prints the tasks included in the downloaded runtime.


In [ ]:
# @title Step 2: Configure runtime paths and show available tasks { display-mode: "form" }

import os
from pathlib import Path

from omegaconf import OmegaConf

AGENTARK_ROOT = Path("/content/agent-ark")
ENV_ROOT = Path("/content/AgentArk-env-1.0.1-linux")
MOD_PATH = ENV_ROOT / "AgentArk_Data" / "Resources" / "Mods"
TASK_STORE_PATH = MOD_PATH / "all_tasks"
RUNTIME_POOL_ROOT = Path("/tmp/agentark_runtime_pool")
VENV_PY = "/content/agentark_env/bin/python"

if not (AGENTARK_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run Step 1 first; /content/agent-ark is missing.")
if not (ENV_ROOT / "AgentArk.x86_64").exists():
    raise RuntimeError("Run Step 1 first; the AgentArk Unity runtime is missing.")

env_values = {
    "AGENTARK_ENV_PATH": str(ENV_ROOT / "AgentArk.x86_64"),
    "AGENTARK_MOD_PATH": str(MOD_PATH),
    "AGENTARK_TASK_STORE_PATH": str(TASK_STORE_PATH),
    "AGENTARK_RUNTIME_TEMPLATE_ROOT": str(ENV_ROOT),
    "AGENTARK_RUNTIME_POOL_ROOT": str(RUNTIME_POOL_ROOT),
    "MLAGENTS_PYTHON_BIN": VENV_PY,
}
os.environ.update(env_values)

(AGENTARK_ROOT / ".env").write_text(
    "\n".join(f"{key}={value}" for key, value in env_values.items()) + "\n",
    encoding="utf-8",
)

mod_config = MOD_PATH / "config.yaml"
if mod_config.exists():
    mod_conf = OmegaConf.load(str(mod_config))
    mod_conf.virtual_display = True
    OmegaConf.save(config=mod_conf, f=str(mod_config))
else:
    print("Warning: mod config not found:", mod_config)

print("Wrote:", AGENTARK_ROOT / ".env")
print("Available tasks:")
task_options = sorted(path.name for path in TASK_STORE_PATH.iterdir() if path.is_dir()) if TASK_STORE_PATH.exists() else []
for name in task_options[:80]:
    print("-", name)
if len(task_options) > 80:
    print(f"... and {len(task_options) - 80} more")
if not task_options:
    print("No tasks found. Check AGENTARK_MOD_PATH and the downloaded runtime.")


## Run Online Model Evaluation

Step 3 configures one OpenAI-compatible model entry under `models`, starts AgentArk, and opens the visualization URL. Set `API_KEY_ENV` to the name of your Colab Secret/environment variable, or fill `API_KEY` only for the current unsaved run.


In [ ]:
# @title Step 3: Run API model evaluation { display-mode: "form" }

TARGET_TASK_NAME = "MarbleStop"  # @param ["AxleBoardAlignment3D", "BlockWorldPathCopy3D", "ColorFovCount", "CraneStackTower2D", "DelayTrain", "DoorSpin3Turns", "FishingJoy2D", "FlappyBird", "GoldMiner2D", "GrenadeClusterCalibration3D", "MarbleStop", "Match3ScoreGoal2D", "MirrorRelay2D", "ObjectRotationMatch", "Pushbox", "Reach2048Tile2D", "RotationSpeedSort3D", "Snake", "SpatialRelationAxisOrder3D", "SpatialRelationMatch3D", "StarterRouteJump3D", "StickBridgeEstimate2D", "Tetris", "TetrisHard", "TightropeSprint3D", "ZigzagPillarJump3D"] {allow-input: true}
GROUP_SEED = 1  # @param {type:"integer"}
ENV_ID = 0  # @param {type:"integer"}
API_NAME = "openrouter-qwen"  # @param {type:"string"}
API_PROVIDER = "openrouter"  # @param {type:"string"}
API_MODEL = "qwen/qwen3.7-plus"  # @param {type:"string"}
API_BASE_URL = "https://openrouter.ai/api/v1"  # @param {type:"string"}
API_KEY_ENV = "OPENROUTER_API_KEY"  # @param {type:"string"}
API_KEY = ""  # @param {type:"string"}
USE_COLAB_SECRET = True  # @param {type:"boolean"}
API_TEMPERATURE = 0.0  # @param {type:"number"}
VIEWER_PORT = 18181  # @param {type:"integer"}

import os
import re
import subprocess
import time
from pathlib import Path

from google.colab import output
from omegaconf import OmegaConf

AGENTARK_ROOT = Path("/content/agent-ark")
CONFIG_PATH = AGENTARK_ROOT / "config" / "ark_env" / "eval_seed1.example.yaml"
VENV_PY = "/content/agentark_env/bin/python"
LOG_PATH = Path("/content/agentark_api_eval.log")

if not os.getenv("AGENTARK_ENV_PATH"):
    raise RuntimeError("Run Step 2 first so AgentArk runtime paths are available.")

if API_KEY and API_KEY_ENV:
    os.environ[API_KEY_ENV] = API_KEY
elif USE_COLAB_SECRET and API_KEY_ENV:
    try:
        from google.colab import userdata

        secret = userdata.get(API_KEY_ENV)
        if secret:
            os.environ[API_KEY_ENV] = secret
            print(f"Loaded API key from Colab Secret: {API_KEY_ENV}")
    except Exception as exc:
        print(f"Colab Secret lookup skipped: {type(exc).__name__}: {exc}")

if API_KEY_ENV and not os.getenv(API_KEY_ENV):
    raise RuntimeError(
        f"Set {API_KEY_ENV} in Colab Secrets, fill API_KEY for this run, or set os.environ[{API_KEY_ENV!r}]."
    )


def ensure_section(parent, name):
    if name not in parent or parent[name] is None:
        parent[name] = {}
    return parent[name]


def safe_slug(value):
    return re.sub(r"[^a-zA-Z0-9]+", "_", value).strip("_").lower() or "task"


conf = OmegaConf.load(str(CONFIG_PATH))
env_cfg = ensure_section(conf, "env_cfg")
env_overrides = ensure_section(env_cfg, "env_config_overrides")
env_overrides.virtual_display = True
env_overrides.num_parallel_envs = 1

eval_cfg = ensure_section(conf, "eval")
eval_cfg.output_path = f"tmp/{safe_slug(TARGET_TASK_NAME)}_{safe_slug(API_NAME)}_seed{int(GROUP_SEED)}.jsonl"
eval_cfg.stop_on_error = True
eval_cfg.fixed_env_id = int(ENV_ID)
eval_cfg.cases = [
    {
        "case_id": f"{safe_slug(TARGET_TASK_NAME)}-api-seed-{int(GROUP_SEED):04d}",
        "task_name": TARGET_TASK_NAME,
        "group_seed": int(GROUP_SEED),
        "env_id": int(ENV_ID),
    }
]

hooks = ensure_section(conf, "hooks")
visualization = ensure_section(hooks, "visualization")
visualization.enabled = True
visualization.host = "127.0.0.1"
visualization.port = int(VIEWER_PORT)
visualization.open_browser = True
visualization.keep_open_on_end = True
human = ensure_section(hooks, "human_interaction")
human.enabled = False

model_config = {
    "name": API_NAME,
    "provider": API_PROVIDER,
    "model": API_MODEL,
    "base_url": API_BASE_URL,
    "temperature": float(API_TEMPERATURE),
}
if API_KEY_ENV:
    model_config["api_key_env"] = API_KEY_ENV
conf.models = [model_config]

OmegaConf.save(config=conf, f=str(CONFIG_PATH))

if "agent_process" in globals() and agent_process.poll() is None:
    print("Stopping previous AgentArk process...")
    agent_process.terminate()
    agent_process.wait(timeout=20)

cmd = [VENV_PY, "-m", "agent_ark.ark_eval.run_api_agent", "--config", "config/ark_env/eval_seed1.example.yaml"]
log_file = LOG_PATH.open("w", encoding="utf-8")
agent_process = subprocess.Popen(
    cmd,
    cwd=str(AGENTARK_ROOT),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy(),
)

time.sleep(3)
if agent_process.poll() is not None:
    print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-5000:])
    raise RuntimeError("AgentArk API evaluation exited early; see the log above.")

print("Evaluation running in background.")
print("Log file:", LOG_PATH)
print("Open visualization:")
print(output.eval_js(f"google.colab.kernel.proxyPort({int(VIEWER_PORT)})"))
